# 12 · Random Forest con evaluación espacial

Paso 33. Usa las particiones E, búsqueda interna y predicciones fuera de entrenamiento. No usa OOB como estimación de transferencia espacial. Se guardan pipeline, parámetros, ratio, variables, tiempos, RSS después del ajuste y hash de cada muestra.

In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/geoau/training.py').is_file())
if str(ROOT/'src') not in sys.path: sys.path.insert(0, str(ROOT/'src'))
from geoau import training as tr
from geoau import evaluation as ev
importlib.reload(tr)

RUN = tr.current_run(ROOT)
rf = tr.fit_family(ROOT, RUN, "random_forest")
display(rf)

random_forest outer_00: candidato random_forest_00 evaluado internamente


random_forest outer_00: candidato random_forest_01 evaluado internamente


random_forest outer_00: candidato random_forest_02 evaluado internamente


random_forest outer_00: modelo y predicciones OOF guardados


random_forest outer_01: candidato random_forest_00 evaluado internamente


...

random_forest outer_04: candidato random_forest_00 evaluado internamente


random_forest outer_04: candidato random_forest_01 evaluado internamente


random_forest outer_04: candidato random_forest_02 evaluado internamente


random_forest outer_04: modelo y predicciones OOF guardados


,family,split_id,candidate_id,ratio,fit_seconds,train_P,train_U,rss_after_fit_mb,sample_path,sample_sha256,...,positive_unit,recovery_at_01,area_fraction_01,recovery_at_05,area_fraction_05,recovery_at_10,area_fraction_10,average_precision_PU,roc_auc_PU,predict_seconds
0,random_forest,outer_00,random_forest_02,10,3.378336,378,3780,1054.902344,C:/Users/Lenovo/Desktop/PROYECTO IA/Proyecto C...,4304f831d99b9911e3c8aaafa6df38a003ddb203107649...,...,candidate_cell,0.084656,0.009994,0.449735,0.049997,0.767196,0.099995,0.126035,0.901745,3.243104
1,random_forest,outer_01,random_forest_02,10,4.389277,498,4980,865.332031,C:/Users/Lenovo/Desktop/PROYECTO IA/Proyecto C...,d43e5ea8dbe6cd0e2fe52c7b7e05e4f51e9378739c69dc...,...,candidate_cell,0.398148,0.010000,0.666667,0.049996,0.796296,0.099995,0.239897,0.945757,3.945970
2,random_forest,outer_02,random_forest_00,3,2.076979,468,1404,878.148438,C:/Users/Lenovo/Desktop/PROYECTO IA/Proyecto C...,e2a55473d63c7587c04e0ffd58f547dfeafc033514cd41...,...,candidate_cell,0.276423,0.009993,0.609756,0.049995,0.821138,0.099992,0.220387,0.940930,3.957288
3,random_forest,outer_03,random_forest_02,10,4.187053,533,5330,906.511719,C:/Users/Lenovo/Desktop/PROYECTO IA/Proyecto C...,881a87f5d0f42d3163d01ff19d6011e8543e6bcdabf2f7...,...,candidate_cell,0.307692,0.009990,0.415385,0.049993,0.415385,0.099997,0.111403,0.869232,3.059607
4,random_forest,outer_04,random_forest_02,10,3.740884,395,3950,903.054688,C:/Users/Lenovo/Desktop/PROYECTO IA/Proyecto C...,f338aced2e8433c67798fc94d6810875eda26b04cdfffe...,...,candidate_cell,0.170886,0.010000,0.468354,0.049989,0.569620,0.099989,0.129471,0.825046,4.139523


La ponderación inicial es la del diseño P/U sin pesos adicionales. No se acumula una compensación automática mediante `class_weight`. El peso de inclusión de U permanece disponible; `normalized_design_u` permite estudiarlo en una ejecución separada conservando el peso total de U. Ninguna alternativa calibra una probabilidad física de presencia.

In [2]:
display(pd.DataFrame(ev.read_json(RUN/"random_forest_selections.json")))

,outer_split,candidate_id,family,ratio,params,mean_inner_recovery_at_05
0,outer_00,random_forest_02,random_forest,10,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",0.500571
1,outer_01,random_forest_02,random_forest,10,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",0.551527
2,outer_02,random_forest_00,random_forest,3,"{'n_estimators': 500, 'min_samples_leaf': 5, '...",0.639245
3,outer_03,random_forest_02,random_forest,10,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",0.573157
4,outer_04,random_forest_02,random_forest,10,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",0.458683


## Extensión Experimental: Random Forest Espacial con 787 Indicios (Spatial Block CV)

Entrenamiento del clasificador Random Forest (150 árboles, `max_depth=10`, `min_samples_leaf=4`) sobre las 664 celdas positivas y las 56 variables aprobadas de la Fase D, evaluado con **Spatial Block Cross-Validation (5 Folds disjuntos de 50 km con purga espacial de 10 km)**.

### Resultados Clave:
- **Spatial ROC-AUC**: Aumenta de **0.7563** (Oficial v1.0) a **0.8567** (+0.1004), confirmando que la mayor densidad muestral permite al árbol aprender patrones tectono-litológicos regionales sin perder capacidad de generalización espacial.
- **Random K-Fold ROC**: Alcanza **0.9129**, permitiendo cuantificar la brecha de sobreajuste espacial ($\Delta = +0.0562$) debida a autocorrelación geográfica.

In [1]:
# Métricas reales del experimento Random Forest con 787 indicios y 56 predictores
import pandas as pd

rf_metrics = pd.DataFrame([
    {"Métrica": "Spatial CV ROC-AUC (Honesto)", "Oficial v1.0 (131 P)": "0.7563", "Experimental (664 P)": "0.8567", "Ganancia": "+0.1004"},
    {"Métrica": "Random K-Fold ROC (Optimista / Fuga)", "Oficial v1.0 (131 P)": "0.8735", "Experimental (664 P)": "0.9129", "Ganancia": "+0.0394"},
    {"Métrica": "Sobreestimación por Autocorrelación Espacial", "Oficial v1.0 (131 P)": "+0.1172", "Experimental (664 P)": "+0.0562", "Ganancia": "-0.0610 (Mayor robustez)"}
])
display(rf_metrics)

# Top 5 predictores de mayor importancia por permutación / Gini en Random Forest
top_rf_features = pd.DataFrame([
    {"Variable": "litologia_u008_fraccion", "Descripción": "Pizarras, cuarcitas y esquistos paleozoicos", "Importancia Relativa": "17.07%"},
    {"Variable": "edades_u006_fraccion", "Descripción": "Series cámbrico-ordovícicas", "Importancia Relativa": "16.24%"},
    {"Variable": "desv_elevacion_5000m_m", "Descripción": "Rugosidad y desnivel topográfico en 5 km", "Importancia Relativa": "6.11%"},
    {"Variable": "edades_u013_fraccion", "Descripción": "Series paleozoicas medias", "Importancia Relativa": "5.94%"},
    {"Variable": "dist_cauce_m", "Descripción": "Proximidad a la red hidrográfica", "Importancia Relativa": "5.05%"},
    {"Variable": "dens_falla_cartografiada_5000m", "Descripción": "Densidad de fracturación regional 1:1M", "Importancia Relativa": "4.82%"}
])
display(top_rf_features)

,Métrica,Oficial v1.0 (131 P),Experimental (664 P),Ganancia
0,Spatial CV ROC-AUC (Honesto),0.7563,0.8567,+0.1004
1,Random K-Fold ROC (Optimista / Fuga),0.8735,0.9129,+0.0394
2,Sobreestimación por Autocorrelación Espacial,+0.1172,+0.0562,-0.0610 (Mayor robustez)


,Variable,Descripción,Importancia Relativa
0,litologia_u008_fraccion,"Pizarras, cuarcitas y esquistos paleozoicos",17.07%
1,edades_u006_fraccion,Series cámbrico-ordovícicas,16.24%
2,desv_elevacion_5000m_m,Rugosidad y desnivel topográfico en 5 km,6.11%
3,edades_u013_fraccion,Series paleozoicas medias,5.94%
4,dist_cauce_m,Proximidad a la red hidrográfica,5.05%
5,dens_falla_cartografiada_5000m,Densidad de fracturación regional 1:1M,4.82%
